# Gold Layer – Business-Curated Healthcare Analytics

The **Gold Layer** represents the final, business-ready data model of the healthcare lakehouse.  
It transforms operational data into curated analytical tables that support decision-making across operations, risk, and finance teams.

This layer provides a **single source of truth for claim lifecycle monitoring**, ensuring transparency from approval to payment settlement.

---

## Objectives of the Gold Layer

- Provide business-consumable datasets for reporting and dashboards  
- Track claim outcomes across operational, fraud, and financial workflows  
- Enable auditability and explainability of claim decisions  
- Support finance reconciliation and risk monitoring  
- Deliver curated tables optimized for BI tools and stakeholder consumption  

---

## Gold Tables Overview

### `Gold_Claim_Decision` – Operational View

This table represents the **adjudication outcome** of each claim.

It answers:
- Is the claim covered under policy?
- What amount is approved?
- Why was the decision made?

Used by:
- Claims Operations Team  
- Audit & Compliance  
- Policy Utilization Analysis  

---

### `Gold_Fraud_Signals` – Risk & Compliance View

This table captures **fraud detection signals** and the current fraud workflow state.

It answers:
- Did the claim trigger any fraud rules?
- How severe is the risk?
- What is the current fraud disposition?

Used by:
- Fraud Investigation Team  
- Compliance Officers  
- Risk Monitoring Dashboards  

---

### `Gold_Reconciliation` – Finance & Settlement View

This table tracks **financial settlement progress** for each claim.

It answers:
- How much was billed vs paid?
- Is the claim fully settled?
- Is payment blocked due to fraud review?
- What action should finance take next?

Used by:
- Finance & Provider Payments Team  
- Cash Flow Monitoring  
- Payment SLA Tracking  

---

## End-to-End Claim Lifecycle Visibility

Together, these Gold tables provide a complete lifecycle view:

-Claim approved by adjudication  
-Fraud engine evaluates risk  
-Finance tracks settlement progress  
-Payment released after clearance  
-Claim moves to closed state  

This ensures all stakeholders can monitor the **same claim from different business perspectives**.

---

## Business Value Delivered

-Transparent claim decisioning  
-Fraud-aware payment control  
-Real-time settlement tracking  
-Reduced revenue leakage  
-Faster audit and compliance reporting  
-Executive-ready analytics datasets  



The Gold layer represents the **final analytical interface** of the healthcare lakehouse, enabling trusted, explainable, and actionable insights across the organization.


In [0]:
from pyspark.sql.functions import *

silver_db = "healthcare.stream_silver"
gold_db   = "healthcare.gold"
chk_base = "/Volumes/healthcare/storage/chk/gold/"

In [0]:
%sql
CREATE OR REPLACE TABLE healthcare.gold.Gold_Claim_Decision AS

WITH line_totals AS (
    SELECT
        claim_id,
        SUM(charge) AS total_charge
    FROM healthcare.stream_silver.claim_line
    GROUP BY claim_id
)

SELECT
    c.claim_id,
    c.patient_id,
    c.provider_id,
    c.claim_date,

    COALESCE(l.total_charge,0) AS total_charge,
    p.coverage_limit,

    -- decision status
    CASE
        WHEN p.policy_id IS NULL THEN 'POLICY_NOT_FOUND'
        WHEN COALESCE(l.total_charge,0) = 0 THEN 'NO_CHARGES'
        WHEN l.total_charge <= p.coverage_limit THEN 'APPROVED'
        WHEN l.total_charge > p.coverage_limit THEN 'PARTIALLY_APPROVED'
    END AS claim_status,

    -- approved amount logic
    CASE
        WHEN p.policy_id IS NULL THEN 0
        WHEN COALESCE(l.total_charge,0) = 0 THEN 0
        WHEN l.total_charge <= p.coverage_limit THEN l.total_charge
        ELSE p.coverage_limit
    END AS approved_amount,

    -- approval ratio useful for dashboards
    CASE
        WHEN COALESCE(l.total_charge,0) = 0 THEN 0
        ELSE ROUND(
            LEAST(l.total_charge, COALESCE(p.coverage_limit,0))
            / l.total_charge * 100, 2
        )
    END AS approval_pct,

    -- decision reason (business readable)
    CASE
        WHEN p.policy_id IS NULL THEN 'Policy not found for claim'
        WHEN COALESCE(l.total_charge,0) = 0 THEN 'No billable procedures'
        WHEN l.total_charge <= p.coverage_limit THEN 'Within coverage limit'
        ELSE 'Exceeded coverage limit'
    END AS decision_reason,

    current_timestamp() AS decision_timestamp

FROM healthcare.stream_silver.claim c
LEFT JOIN line_totals l
    ON c.claim_id = l.claim_id
LEFT JOIN healthcare.stream_silver.policy p
    ON c.policy_id = p.policy_id;


In [0]:
%sql
CREATE OR REPLACE TABLE healthcare.gold.Gold_Fraud_Signals AS

WITH fraud_config AS (
    SELECT 
        15000 AS high_amount_threshold,
        'High Claim Amount' AS detection_rule
),

line_totals AS (
    SELECT
        claim_id,
        SUM(charge) AS total_charge
    FROM healthcare.stream_silver.claim_line
    GROUP BY claim_id
)

SELECT
    c.claim_id,
    c.provider_id,
    c.patient_id,

    COALESCE(l.total_charge, 0) AS total_charge,

    -- detection rule
    CASE
        WHEN COALESCE(l.total_charge,0) > f.high_amount_threshold
        THEN f.detection_rule
        ELSE 'No Risk Rule Triggered'
    END AS detection_rule,

    -- suspicious flag
    CASE
        WHEN COALESCE(l.total_charge,0) > f.high_amount_threshold
        THEN 'YES'
        ELSE 'NO'
    END AS suspicious_flag,

    -- risk severity
    CASE
        WHEN COALESCE(l.total_charge,0) > f.high_amount_threshold * 2 THEN 'CRITICAL'
        WHEN COALESCE(l.total_charge,0) > f.high_amount_threshold THEN 'HIGH'
        ELSE 'LOW'
    END AS risk_severity,

    -- NEW FRAUD DISPOSITION COLUMN
    CASE
        WHEN COALESCE(l.total_charge,0) > f.high_amount_threshold 
            THEN 'UNDER_REVIEW'
        ELSE 'AUTO_CLEAR'
    END AS fraud_disposition,

    current_timestamp() AS detection_timestamp

FROM healthcare.stream_silver.claim c
LEFT JOIN line_totals l
    ON c.claim_id = l.claim_id
CROSS JOIN fraud_config f;


In [0]:
%sql
CREATE OR REPLACE TABLE healthcare.gold.Gold_Reconciliation AS

WITH billed AS (
    SELECT
        claim_id,
        SUM(charge) AS billed_amount
    FROM healthcare.stream_silver.claim_line
    GROUP BY claim_id
),

paid AS (
    SELECT
        claim_id,
        SUM(amount) AS paid_amount,
        MAX(ingestion_ts) AS last_payment_ts
    FROM healthcare.stream_silver.payment
    GROUP BY claim_id
),

fraud AS (
    SELECT
        claim_id,
        suspicious_flag
    FROM healthcare.gold.Gold_Fraud_Signals
)

SELECT
    c.claim_id,

    -- financial values
    COALESCE(b.billed_amount, 0) AS billed_amount,
    COALESCE(p.paid_amount, 0) AS paid_amount,
    COALESCE(b.billed_amount, 0) - COALESCE(p.paid_amount, 0) AS variance,

    -- payment completion %
    CASE
        WHEN COALESCE(b.billed_amount,0)=0 THEN 0
        ELSE ROUND(COALESCE(p.paid_amount,0)/b.billed_amount*100,2)
    END AS payment_completion_pct,

    -- reconciliation status
    CASE
        WHEN p.paid_amount IS NULL THEN 'PENDING_PAYMENT'
        WHEN p.paid_amount < b.billed_amount THEN 'PARTIALLY_PAID'
        WHEN p.paid_amount = b.billed_amount THEN 'FULLY_PAID'
        WHEN p.paid_amount > b.billed_amount THEN 'OVERPAID'
    END AS recon_status,

    -- finance action
    CASE
        WHEN p.paid_amount IS NULL THEN 'OPEN_CLAIM'
        WHEN p.paid_amount < b.billed_amount THEN 'FINANCE_REVIEW'
        WHEN p.paid_amount > b.billed_amount THEN 'REFUND_REQUIRED'
        ELSE 'SETTLED'
    END AS finance_action,

    -- FRAUD HOLD FLAG (new column)
    CASE
        WHEN f.suspicious_flag = 'YES' THEN 'YES'
        ELSE 'NO'
    END AS fraud_hold_flag,

    -- timestamp kept intact
    p.last_payment_ts,

    -- business explanation for dashboards
    CASE
        WHEN p.last_payment_ts IS NULL 
            THEN 'No payment has been received from the payer yet'
        ELSE 'Payment received'
    END AS payment_status_message,

    current_timestamp() AS reconciliation_timestamp

FROM healthcare.stream_silver.claim c
LEFT JOIN billed b ON c.claim_id = b.claim_id
LEFT JOIN paid p ON c.claim_id = p.claim_id
LEFT JOIN fraud f ON c.claim_id = f.claim_id;


In [0]:
%sql
select * from  healthcare.gold.Gold_Claim_Decision  where claim_id=2250

In [0]:
%sql
select * from healthcare.gold.Gold_Fraud_Signals where claim_id=2250

In [0]:
%sql
select * from healthcare.gold.Gold_Reconciliation where claim_id=2250
